In [1]:
import os
os.chdir("../../")#与easymlops同级目录

### 拆分数据

In [2]:
import pandas as pd
data=pd.read_csv("./data/demo.csv",encoding="gbk").sample(frac=1)
data.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
467,468,0,1,"Smart, Mr. John Montgomery",male,56.0,0,0,113792,26.5500,NaN,S
542,543,0,3,"Andersson, Miss. Sigrid Elisabeth",female,11.0,4,2,347082,31.2750,NaN,S
764,765,0,3,"Eklund, Mr. Hans Linus",male,16.0,0,0,347074,7.7750,NaN,S
570,571,1,2,"Harris, Mr. George",male,62.0,0,0,S.W./PP 752,10.5000,NaN,S
74,75,1,3,"Bing, Mr. Lee",male,32.0,0,0,1601,56.4958,NaN,S


In [3]:
x_train=data[:500]
x_test=data[500:]
y_train=x_train["Survived"]
y_test=x_test["Survived"]
del x_train["Survived"]
del x_test["Survived"]

### 简单跑一个模型

In [4]:
from easymlops import TablePipeLine
from easymlops.table.preprocessing import *

In [5]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())

table.fit(x_train)

### 挂载

In [6]:
from easymlops.table.storage import ElasticSearchStorage

In [7]:
table[-1].set_branch_pipe(ElasticSearchStorage(index_name="survived_index",es_args=["http://192.168.244.133:9200"],es_kwargs=dict(timeout=60)))

In [8]:
table[-1].get_branch_pipe(0).fit(table.transform(x_train,run_to_layer=-1))

<ElasticSearchStorage(<class 'easymlops.table.storage.elasticsearch_storage.ElasticSearchStorage'>, started 11104)>

### 模拟线上数据流

In [9]:
i=0
for record in tqdm(x_test[:10].to_dict("record")):
    table.transform_single(record,storage_base_dict={"key":i})
    i+=1

100%|████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 3332.78it/s]


### 查询key

In [10]:
table[-1].get_branch_pipe(0).select_key(key="3")

,storage_key,storage_transform_time,Age,Cabin,Embarked,Fare,Name,Parch,PassengerId,Pclass,Sex,SibSp,Ticket
0,3,2023-03-25 09:06:39.684274,0.0,nan,Q,7.734375,"O'Connell, Mr. Patrick D",0.0,630.0,3.0,male,0.0,334912


### query搜索 
前三条最相似的记录

In [12]:
body = {
    'query': {
        'match': {
            'Name': 'Lester, Mr. James'
        }
    },
    'sort': {
        '_score': {                    
            'order': 'desc'        
        }
    },
    'from':0,
    'size':3
}

In [13]:
table[-1].get_branch_pipe(0).query(body=body)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,storage_key,storage_transform_time,hit_score_
0,811.0,3.0,"Alexander, Mr. William",male,26.0,0.0,0.0,3474,7.886719,nan,S,0,2023-03-25 09:06:39.683273,0.637226
1,759.0,3.0,"Theobald, Mr. Thomas Leonard",male,34.0,0.0,0.0,363294,8.046875,nan,S,default,2023-03-25 08:49:59.711673,0.573403
2,28.0,1.0,"Fortune, Mr. Charles Alexander",male,19.0,3.0,2.0,19950,263.000000,C23 C25 C27,S,2,2023-03-25 09:06:39.684274,0.573403


### query聚合  
Pclass分组下最大的Fare

In [14]:
body = {
    '_source':['Pclass','Fare'],
    'aggs': {
        'Pclass': {
            'max': {
                'field':'Fare'
            }
        }
    },
    'sort': {
        'Pclass': {                    
            'order': 'asc'        
        }
    },
    'from':0,
    'size':3
}

In [15]:
table[-1].get_branch_pipe(0).query(body=body)

,Pclass,Fare
0,1.0,153.5000
1,1.0,263.0000
2,1.0,77.3125


### search透传

In [16]:
table[-1].get_branch_pipe(0).search(body=body)

{'took': 3,
 'timed_out': False,
 '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0},
 'hits': {'total': {'value': 12, 'relation': 'eq'},
  'max_score': None,
  'hits': [{'_index': 'survived_index',
    '_type': '_doc',
    '_id': '1',
    '_score': None,
    '_source': {'Pclass': 1, 'Fare': 153.5},
    'sort': [1]},
   {'_index': 'survived_index',
    '_type': '_doc',
    '_id': '2',
    '_score': None,
    '_source': {'Pclass': 1, 'Fare': 263.0},
    'sort': [1]},
   {'_index': 'survived_index',
    '_type': '_doc',
    '_id': '4',
    '_score': None,
    '_source': {'Pclass': 1, 'Fare': 77.3125},
    'sort': [1]}]},
 'aggregations': {'Pclass': {'value': 263.0}}}

### 异步保存
存储模块强制使用异步的方式存储数据，通常只消耗少量transform_single的时间(获取当前时间以及字典中的数据，并放到队列中异步存储)

In [17]:
from easymlops.table.callback import check_transform_function_pipeline
table.callback(check_transform_function_pipeline,x_test[:10])

(<class 'easymlops.table.preprocessing.core.FixInput'>) module check [transform] complete,speed:[0.2ms]/it,cpu:[0%],memory:[0K]
(<class 'easymlops.table.preprocessing.onevar_operation.FillNa'>) module check [transform] complete,speed:[0.1ms]/it,cpu:[0%],memory:[0K]


### 持久化测试

In [14]:
table.save("ml.pkl")

In [15]:
table=TablePipeLine()
table.pipe(FixInput())\
  .pipe(FillNa())

table[-1].set_branch_pipe(ElasticSearchStorage())
table.load("ml.pkl")

In [16]:
table[-1].get_branch_pipe(0).search(body=body)

,storage_key,storage_transform_time,Age,Cabin,Embarked,Fare,Name,Parch,PassengerId,Pclass,Sex,SibSp,Ticket
0,3,2023-03-25 08:49:20.252347,39.0,nan,S,24.156250,"Lester, Mr. James",0.0,812.0,3.0,male,0.0,A/4 48871
1,4,2023-03-25 08:49:20.252347,19.0,nan,S,8.156250,"Crease, Mr. Ernest James",0.0,68.0,3.0,male,0.0,S.P. 3464
2,2,2023-03-25 08:49:20.251337,28.0,nan,S,7.894531,"Hendekovic, Mr. Ignjac",0.0,314.0,3.0,male,0.0,349243
